# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library. We show how to access the dataset, browse metadata, and perform basic data analysis—all referencing entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant Schema URL:**
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema describes entities using `@id`. We will enumerate the record sets and fields using their `@id` for subsequent referencing.

In [ ]:
# List all record sets and their fields using @id
record_sets = []
for rs in dataset.metadata.recordSet:
    print(f"Record Set: {rs['@id']} | Name: {rs.get('name','(Unnamed)')}")
    fields = rs.get('field',[])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  Field: {field['@id']} | Name: {field.get('name','(Unnamed)')}")
    record_sets.append(rs['@id'])

# As a quick preview, print all records from the first record set
if record_sets:
    first_record_set_id = record_sets[0]
    print(f"\nPreview of records in record set {first_record_set_id}:")
    for x in dataset.records(record_set=first_record_set_id):
        print(x)
        break  # print only the first record as a sample

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nData for Record Set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())

# Select a record set for further analysis
selected_record_set_id = record_sets[0] if record_sets else Nonedf = dataframes.get(selected_record_set_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

All references are via `@id`.

In [ ]:
# Inspect column names and select fields by their @id
print("Columns in selected record set:", df.columns.tolist())

# Example: Let's suppose 'gad7_total_score' and 'village' are columns in the DataFrame
# Their @id could be something like 'https://api.app.sen.science/frontiers/7160411/gad7_total_score' and 'https://api.app.sen.science/frontiers/7160411/village'
# For demonstration, adapt the column selection if actual column names differ

numeric_field_id = None
for col in df.columns:
    # Search for 'gad7', 'phq9', or 'psq' scores, or other numeric fields
    if 'gad7' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id:
    threshold = 10
    # Convert field to numeric (important for real-world datasets)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field such as village
    group_field_id = None
    for col in df.columns:
        if 'village' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Ensure that all fields referenced are by their `@id`.

In [ ]:
# Example: Visualize the distribution of the selected numeric field
if numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists, show the grouped bar plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=grouped_df,
            x=group_field_id,
            y=numeric_field_id
        )
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County using `mlcroissant`.
- Entities were always referenced by their `@id`.
- Record sets and fields were browsed interactively, and numeric indicators (e.g., GAD-7 score) were visualized.
- Further analysis can be performed by referencing other fields and entities by `@id` and applying similar processing and visualization steps.

For more details, please refer to the [FAIR² dataset publication](https://sen.science/doi/10.71728/senscience.vcs2-05nj) and [mlcroissant documentation](https://mlcommons.org/croissant/).
